# CSC14120 – PARALLEL PROGRAMMING: FINAL PROJECT
## Midterm Submission: Proposal and Sequential Version (`V0`)

---

## 3.1 Team Information and Work Plan

### Team Members
| No. | Full Name | Student ID | Role |
| :--- | :--- | :--- | :--- |
| 1 | *[Điền tên thành viên 1]* | *[MSSV 1]* | Team Leader / Stages 1–3 & V0, V2 |
| 2 | *[Điền tên thành viên 2]* | *[MSSV 2]* | Member / Stages 4–6 & V1, V3 |
| 3 | *[Điền tên thành viên 3 (nếu có)]* | *[MSSV 3]* | Member / Dataset, Profiling & V4 |

### Work Plan & Responsibility Matrix (Midterm $\rightarrow$ Final)
| Pipeline Stage / Version | Responsible Member(s) | Target Milestone |
| :--- | :--- | :--- |
| **Dataset Capture & Preparation** (3 sites + 1 challenging pair, 3 resolutions) | All Members | Midterm (Week 7) |
| **V0: Sequential C++ Baseline** (All 6 stages + Timing + Verification) | All Members | Midterm (Week 7) |
| **V1: Basic CUDA** (Naive per-pixel / per-keypoint / per-hypothesis kernels) | Member 1, Member 2 | Week 9 |
| **V2: Memory & Stencil Optimizations** (Separable Gaussian, Shared Memory Tiling + Halo, Constant Memory) | Member 1 | Week 10 |
| **V3: Matching & Warp Intrinsics** (Tiled Hamming Matching with `__popc`, Warp-Shuffle RANSAC) | Member 2 | Week 11 |
| **V4: Pipeline Optimizations** (Multi-Stream Image A/B concurrency, Pinned Memory, Texture Warping) | Member 3 | Week 12 |
| **Final Report, Video & Oral Defense** | All Members | Final Submission |


---
## 3.2 Problem Understanding

**Panorama Image Stitching** takes two overlapping photographs of the same scene—captured by rotating a camera around its optical center—and combines them into a single, seamless wide-angle composite image.

The complete pipeline consists of **six sequential stages**:
1. **Preprocessing (Independent per image):** Converts each input RGB image into a single-channel luminance (grayscale) image and applies a Gaussian low-pass filter to suppress high-frequency sensor noise before gradient computation.
2. **Keypoint Detection (Independent per image):** Locates salient, repeatable 2D structures (corners) where intensity changes strongly in two orthogonal directions, suppresses local duplicates via Non-Maximum Suppression (NMS), and selects the strongest Top-$K$ keypoints.
3. **Feature Description (Independent per image):** Extracts a compact, orientation-normalized binary signature around each keypoint so that corresponding physical points across the two views have similar descriptors despite slight camera rotation and illumination changes.
4. **Feature Matching (Cross-image):** Compares every descriptor in Image A against every descriptor in Image B, retaining only unambiguous correspondences that pass both Lowe's second-nearest-neighbor ratio test and mutual cross-checking.
5. **Homography Estimation (Cross-image):** Estimates the $3 \times 3$ projective transformation matrix $\mathbf{H}$ mapping coordinates in Image B to Image A ($\mathbf{p}_A \sim \mathbf{H}\,\mathbf{p}_B$). Because raw feature matches inevitably contain outliers, **RANSAC** repeatedly samples minimal 4-point subsets, solves the Normalized Direct Linear Transform (DLT), and refines $\mathbf{H}$ over the largest consensus set (inliers).
6. **Warping and Blending (Cross-image):** Computes the bounding canvas of the composite panorama, applies **inverse warping** ($\mathbf{H}^{-1}$) with bilinear interpolation to sample Image B without holes, and blends the overlapping region with distance-based linear feathering to eliminate visible seams.


---
## 3.3 Chosen Methods and Justification

| Stage | Chosen Method | Key Parameters | Justification (Output Quality & GPU Suitability) |
| :--- | :--- | :--- | :--- |
| **1. Preprocessing** | Luminance Grayscale + 2D Gaussian Blur | Kernel $5\times 5$ ($r=2$), $\sigma = 1.4$ | Effectively removes aliasing and sensor noise prior to Sobel differentiation. Regular 2D stencil is ideal for GPU thread blocks, shared-memory halo tiling, and 1D separable decomposition in V2. |
| **2. Keypoint Detection** | **Harris Corner Detector** ($3\times 3$ Sobel + $3\times 3$ tensor box filter) + NMS + Top-$K$ | $k = 0.04$, threshold $= 10^4$, NMS window $5\times 5$ ($r=2$), $K = 3000$, border $= 16\text{ px}$ | Produces highly accurate, well-distributed corners on architectural/natural scenes. Unlike FAST (which suffers from heavy warp divergence) or multi-octave DoG (complex scale-space memory management), Harris consists of uniform per-pixel stencil operations followed by parallel compaction/sorting. |
| **3. Description** | **Rotated BRIEF (rBRIEF 256-bit)** with Intensity Centroid orientation | Patch $31\times 31$ ($r=15$), 256 point pairs (`8 × uint32_t`), fixed seed `42` | Adding intensity-centroid angle $\theta = \text{atan2}(m_{01}, m_{10})$ provides rotation invariance for handheld camera pans while keeping each descriptor at just **32 bytes** (16× smaller than 128-float SIFT). Sampling offsets fit entirely in GPU `__constant__` memory. |
| **4. Matching** | **Brute-Force Hamming Distance** + **Lowe's Ratio Test** + **Mutual Cross-Check** | Ratio threshold $= 0.75$, Max Hamming dist $= 64$ | Combining Lowe's ratio test with mutual cross-check yields a >80% inlier ratio. On the GPU, all-pairs $N_A \times N_B$ binary matching maps directly to shared-memory tiled matrix multiplication using hardware `__popc(a ^ b)` instructions. |
| **5. Homography** | **RANSAC** + **Hartley-Normalized 4-Point DLT** + **Inlier Least-Squares Refinement** | $2000$ iterations, inlier threshold $= 3.0\text{ px}$, fixed seed `42` | Hartley normalization prevents numerical ill-conditioning at 4K coordinates (`3840×2160`). Solving the $8\times 8$ normal equations via Gaussian elimination with partial pivoting avoids heavy general SVD libraries and allows each GPU thread/warp to evaluate an independent RANSAC hypothesis. |
| **6. Warping & Blending** | **Inverse Warping** + **Bilinear Interpolation** + **Distance-to-Border Linear Feathering** | Clamped canvas bounds, bilinear subpixel sampling, $\alpha = d_A / (d_A + d_B)$ | Inverse mapping guarantees zero holes in the output panorama, and linear feathering removes exposure seams cleanly. Every output pixel $(x_{out}, y_{out})$ is independent, mapping naturally to 2D GPU grids and hardware texture units. |


---
## 3.4 Sequential Version (`V0`) Design & Build Setup

The sequential baseline `V0` is implemented in `src/v0_sequential.cpp` and compiled alongside `src/main.cu` and `src/stb_impl.cpp` (one source of truth):
- **Strict Baseline Fairness:** Runs on a single CPU thread with no OpenMP, no `std::thread`, and no hand-written SIMD intrinsics.
- **Key Design Excerpt — Hartley-Normalized DLT & Inlier Refinement (`src/v0_sequential.cpp`):**
```cpp
// Normalize coordinates to zero mean and mean distance sqrt(2) before solving 8x8 DLT:
const double uA = (kpA[m.idxA].x - mean_xA) * scale_A;
const double vA = (kpA[m.idxA].y - mean_yA) * scale_A;
const double uB = (kpB[m.idxB].x - mean_xB) * scale_B;
const double vB = (kpB[m.idxB].y - mean_yB) * scale_B;
const double r1[8] = { uB, vB, 1.0, 0.0, 0.0, 0.0, -uA * uB, -uA * vB };
const double r2[8] = { 0.0, 0.0, 0.0, uB, vB, 1.0, -vA * uB, -vA * vB };
```

Below, we compile the project directly from `src/` and generate the synthetic verification benchmark across all three required resolutions (`1280×720`, `1920×1080`, `3840×2160`).

In [1]:
# Build executable from src/ and generate synthetic ground-truth benchmark pairs
!mkdir -p dataset output
!make
!g++ -O3 -std=c++17 src/gen_synthetic.cpp src/stb_impl.cpp -o gen_synthetic && ./gen_synthetic && rm -f gen_synthetic

Compiling with /opt/cuda/bin/nvcc...
Generated dataset/synthetic_small (1280x720): dataset/synthetic_small_left.png, dataset/synthetic_small_right.png, dataset/synthetic_small_gt.txt
Generated dataset/synthetic_medium (1920x1080): dataset/synthetic_medium_left.png, dataset/synthetic_medium_right.png, dataset/synthetic_medium_gt.txt
Generated dataset/synthetic_large (3840x2160): dataset/synthetic_large_left.png, dataset/synthetic_large_right.png, dataset/synthetic_large_gt.txt


---
## 3.5 `V0` Results and Profiling

### 3.5.1 Benchmark Execution Across Three Resolutions (Small, Medium, Large)

In [2]:
!./stitch dataset/synthetic_small_left.png dataset/synthetic_small_right.png output/pano_small.png --version 0 --runs 5 --gt dataset/synthetic_small_gt.txt

GPU Device : None (CPU execution)
Input A    : dataset/synthetic_small_left.png (1280 x 720)
Input B    : dataset/synthetic_small_right.png (1280 x 720)
Version    : V0 (5 timed runs + 1 warm-up run)

--- Pipeline Statistics ---
Keypoints (Image A / Image B) : 3000 / 3000
Matched Keypoint Pairs        : 817
RANSAC Inliers                : 698
Output Panorama Size          : 1761 x 789
Estimated Homography H (B -> A):
  [     0.999682     -0.035157    448.007102 ]
  [     0.035094      0.999410     14.371515 ]
  [    -0.000015      0.000010      1.000000 ]

--- Ground-Truth Homography Verification ---
Frobenius Norm Error ||H_est - H_gt||_F : 0.029359
Mean Point Reprojection Error           : 0.0968 px
Max  Point Reprojection Error           : 0.2181 px

--- Stage Timing Breakdown (Average of 5 runs) ---
Stage                          |     Avg (ms) |     Min (ms) | Share (%)
------------------------------------------------------------------------
1. Preprocessing               |       

In [3]:
!./stitch dataset/synthetic_medium_left.png dataset/synthetic_medium_right.png output/pano_medium.png --version 0 --runs 5 --gt dataset/synthetic_medium_gt.txt

GPU Device : None (CPU execution)
Input A    : dataset/synthetic_medium_left.png (1920 x 1080)
Input B    : dataset/synthetic_medium_right.png (1920 x 1080)
Version    : V0 (5 timed runs + 1 warm-up run)

--- Pipeline Statistics ---
Keypoints (Image A / Image B) : 3000 / 3000
Matched Keypoint Pairs        : 655
RANSAC Inliers                : 508
Output Panorama Size          : 2642 x 1183
Estimated Homography H (B -> A):
  [     0.999052     -0.035006    672.076057 ]
  [     0.034891      0.999359     21.617641 ]
  [    -0.000010      0.000007      1.000000 ]

--- Ground-Truth Homography Verification ---
Frobenius Norm Error ||H_est - H_gt||_F : 0.078077
Mean Point Reprojection Error           : 0.1150 px
Max  Point Reprojection Error           : 0.1977 px

--- Stage Timing Breakdown (Average of 5 runs) ---
Stage                          |     Avg (ms) |     Min (ms) | Share (%)
------------------------------------------------------------------------
1. Preprocessing               |  

In [4]:
!./stitch dataset/synthetic_large_left.png dataset/synthetic_large_right.png output/pano_large.png --version 0 --runs 5 --gt dataset/synthetic_large_gt.txt

GPU Device : None (CPU execution)
Input A    : dataset/synthetic_large_left.png (3840 x 2160)
Input B    : dataset/synthetic_large_right.png (3840 x 2160)
Version    : V0 (5 timed runs + 1 warm-up run)

--- Pipeline Statistics ---
Keypoints (Image A / Image B) : 3000 / 3000
Matched Keypoint Pairs        : 410
RANSAC Inliers                : 259
Output Panorama Size          : 5284 x 2365
Estimated Homography H (B -> A):
  [     0.999497     -0.034958   1343.931056 ]
  [     0.035061      0.999480     43.070711 ]
  [    -0.000005      0.000003      1.000000 ]

--- Ground-Truth Homography Verification ---
Frobenius Norm Error ||H_est - H_gt||_F : 0.146523
Mean Point Reprojection Error           : 0.1478 px
Max  Point Reprojection Error           : 0.2809 px

--- Stage Timing Breakdown (Average of 5 runs) ---
Stage                          |     Avg (ms) |     Min (ms) | Share (%)
------------------------------------------------------------------------
1. Preprocessing               |    

### 3.5.2 Summary Analysis of `V0` Correctness, Timings, Hot Stages & Amdahl's Law

#### 1. Correctness on Synthetic Ground-Truth Pairs
Across all three resolutions, the Hartley-normalized 4-point DLT + Inlier Least-Squares Refinement recovers the ground-truth homography $\mathbf{H}_{gt}$ with **sub-pixel accuracy**:
- **Small (`1280×720`):** Frobenius error $= 0.029359$, Mean point reprojection error $= 0.0968\text{ px}$ (Max $= 0.2181\text{ px}$), $698 / 817$ inliers ($85.4\%$).
- **Medium (`1920×1080`):** Frobenius error $= 0.078077$, Mean point reprojection error $= 0.1150\text{ px}$ (Max $= 0.1977\text{ px}$), $508 / 655$ inliers ($77.6\%$).
- **Large (`3840×2160`):** Frobenius error $= 0.146523$, Mean point reprojection error $= 0.1478\text{ px}$ (Max $= 0.2809\text{ px}$), $259 / 410$ inliers ($63.2\%$).

#### 2. Per-Stage Timing Summary Table (Average of 5 Timed Runs)
| Stage | Small `1280×720` (ms / %) | Medium `1920×1080` (ms / %) | Large `3840×2160` (ms / %) | Scaling Behavior |
| :--- | ---: | ---: | ---: | :--- |
| **1. Preprocessing** | `73.716 ms` (`13.81%`) | `160.185 ms` (`23.16%`) | `668.949 ms` (`41.68%`) | $O(W \times H)$ — grows $9.07\times$ from 720p to 4K |
| **2. Keypoint Detection** | `19.132 ms` (`3.58%`) | `35.546 ms` (`5.14%`) | `120.657 ms` (`7.52%`) | $O(W \times H)$ — grows $6.31\times$ from 720p to 4K |
| **3. Description (rBRIEF)** | `60.194 ms` (`11.27%`) | `59.912 ms` (`8.66%`) | `68.430 ms` (`4.26%`) | $O(K \cdot r^2)$ — nearly constant (~60–68 ms) for $K=3000$ |
| **4. Feature Matching** | `318.765 ms` (`59.70%`) | `318.408 ms` (`46.04%`) | `329.800 ms` (`20.55%`) | $O(K^2)$ — constant (~318–330 ms) for $K=3000$ |
| **5. Homography (RANSAC)** | `15.407 ms` (`2.89%`) | `12.930 ms` (`1.87%`) | `7.861 ms` (`0.49%`) | $O(I_{\text{ransac}} \cdot M)$ — proportional to match count $M$ |
| **6. Warping & Blending** | `46.689 ms` (`8.74%`) | `104.553 ms` (`15.12%`) | `409.421 ms` (`25.51%`) | $O(W_{out} \times H_{out})$ — grows $8.77\times$ from 720p to 4K |
| **Total End-to-End** | **`533.905 ms` (`100%`)** | **`691.535 ms` (`100%`)** | **`1605.119 ms` (`100%`)** | **$3.01\times$ overall increase from 720p to 4K** |

#### 3. Hot Stages & Shift Across Resolutions
- **At Small (`1280×720`) and Medium (`1920×1080`) resolutions:** **Stage 4 (Feature Matching)** is the primary bottleneck, consuming **`59.70%`** (`318.765 ms`) at 720p and **`46.04%`** (`318.408 ms`) at 1080p, because all-pairs Hamming matching evaluates $3000 \times 3000 = 9 \times 10^6$ descriptor pairs in both forward and cross-check directions.
- **At Large (`3840×2160` - 4K) resolution:** Because the number of keypoints is capped at $K = 3000$, Stage 3 and Stage 4 remain constant in absolute time, whereas the pixel-parallel stages scale linearly with the $9\times$ increase in pixel count. Consequently, **Stage 1 (Preprocessing: `668.949 ms`, `41.68%`)** and **Stage 6 (Warping & Blending: `409.421 ms`, `25.51%`)** overtake Feature Matching (`329.800 ms`, `20.55%`) as the dominant stages.

#### 4. Amdahl's Law Speedup Limit
- **Full 6-Stage GPU Acceleration Plan:** Since we plan to accelerate **all 6 stages** on the GPU (leaving only negligible host-side launch overhead $<0.01\%$), the parallelizable fraction is $p > 0.999$, meaning Amdahl's Law does not bottleneck our pipeline on computation (end-to-end speedup will instead be bounded by PCIe Host$\leftrightarrow$Device transfer time).
- **Even if Stage 5 (RANSAC) were kept on CPU** and only Stages 1, 2, 3, 4, and 6 were moved to the GPU:
  - **Small (`1280×720`):** $p = 0.9711 \implies \text{Speedup}_{\max} = \frac{1}{1 - 0.9711} = \mathbf{34.65\times}$
  - **Medium (`1920×1080`):** $p = 0.9813 \implies \text{Speedup}_{\max} = \frac{1}{1 - 0.9813} = \mathbf{53.48\times}$
  - **Large (`3840×2160`):** $p = 0.9951 \implies \text{Speedup}_{\max} = \frac{1}{1 - 0.9951} = \mathbf{204.15\times}$

---
## 3.6 Dataset

- **Dataset Google Drive Link:** *[Chèn link Google Drive chứa ảnh gốc và thư mục `dataset/` của nhóm tại đây]*

| Pair ID | Site Description | Scene Type | Overlap | Resolutions Prepared | Purpose |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Site 1** | *[Ví dụ: Thư viện / Phòng học ĐH KHTN]* | Indoor | ~40% | `1280×720`, `1920×1080`, `3840×2160` | Standard pair 1 |
| **Site 2** | *[Ví dụ: Sân trường ĐH KHTN cơ sở Linh Trung]* | Outdoor | ~40% | `1280×720`, `1920×1080`, `3840×2160` | Standard pair 2 |
| **Site 3** | *[Ví dụ: Khu vực phố đi bộ / tòa nhà]* | Team's Choice | ~45% | `1280×720`, `1920×1080`, `3840×2160` | Standard pair 3 |
| **Challenging** | *[Ví dụ: Hành lang tường trơn / gạch lặp lại / cảnh đêm]* | Challenging | ~35% | `1280×720`, `1920×1080`, `3840×2160` | Failure case analysis |


---
## 3.7 GPU Plan

### 1. Stages Selected for GPU Acceleration
Based on the profiling results in Section 3.5, **all 6 stages** will be moved to the GPU:
- **Stages 1, 2, and 6** account for **74.71%** of the runtime at 4K (`3840×2160`) and exhibit massive data parallelism across millions of pixels.
- **Stages 3 and 4** account for **70.97%** of the runtime at 720p (`1280×720`) and **54.70%** at 1080p (`1920×1080`), where $O(K^2)$ descriptor comparisons dominate.
- **Stage 5 (RANSAC)** is also moved to the GPU so that all intermediate data (keypoints, descriptors, matches, and homography) remains resident in GPU device memory from Stage 1 through Stage 6 without costly intermediate Host$\leftrightarrow$Device round-trips.

### 2. Planned Thread Mapping for `V1` (Basic CUDA)
| Stage | Grid / Block Configuration | Work Assigned per Thread |
| :--- | :--- | :--- |
| **1. Preprocessing** | 2D Grid of `16×16` blocks over $(W, H)$ | **1 thread per pixel $(x, y)$:** converts RGB to grayscale and computes the $5\times 5$ 2D Gaussian convolution from global memory. |
| **2. Keypoint Detection** | 2D Grid of `16×16` blocks over $(W, H)$ | **1 thread per pixel $(x, y)$:** Kernel 1 computes Sobel gradients $I_x^2, I_y^2, I_x I_y$; Kernel 2 computes $3\times 3$ Harris response $R$; Kernel 3 performs $5\times 5$ NMS and appends surviving keypoints via `atomicAdd` on a global counter, followed by GPU Thrust/radix sort for Top-$K$. |
| **3. Description (rBRIEF)** | 1D Grid of `256` threads over $K$ keypoints | **1 thread per keypoint:** computes the intensity centroid orientation $\theta$ over the $31\times 31$ patch and packs the 256 rotated bit comparisons into `uint32_t bits[8]`. |
| **4. Feature Matching** | 1D Grid of `256` threads over $N_A$ keypoints | **1 thread per keypoint in Image A:** loops over all $N_B$ descriptors in Image B in global memory, tracks `best_d` and `second_d`, and applies Lowe's ratio test + cross-check. |
| **5. Homography (RANSAC)** | 1D Grid of `256` threads over `2000` iterations | **1 thread per RANSAC hypothesis:** loads 4 pre-generated random match indices, solves the $8\times 8$ normalized DLT system in local registers, counts inliers across all matches, and updates the best hypothesis index via `atomicMax`. |
| **6. Warping & Blending** | 2D Grid of `16×16` blocks over $(W_{out}, H_{out})$ | **1 thread per output panorama pixel $(x_{out}, y_{out})$:** multiplies by $\mathbf{H}^{-1}$, performs bilinear interpolation on Image B, and blends with Image A using linear feathering weights. |

### 3. Planned Optimizations for `V2`, `V3`, `V4` and Foreseen Risks
1. **Optimization 1 (`V2` – Separable Convolution + Shared Memory Halo Tiling + Constant Memory):**
   - *Idea:* Decompose the $5\times 5$ 2D Gaussian blur (25 MACs/pixel) into horizontal and vertical 1D passes ($5 + 5 = 10$ MACs/pixel), cache image tiles with halos in `__shared__` memory for both Gaussian blur and Harris Sobel/box filters, and place filter weights and the 256 BRIEF sampling pairs in `__constant__` memory.
   - *Risk:* Shared-memory bank conflicts or halo loading divergence at block boundaries if tile padding and apron dimensions are not aligned to 32-thread warps.
2. **Optimization 2 (`V3` – Shared-Memory Tiled Matching with `__popc` + Warp-Shuffle RANSAC):**
   - *Idea:* Tile descriptors of Image B into `__shared__` memory (analogous to tiled GEMM) so each descriptor is loaded once per thread block rather than $N_A$ times from global memory, use the hardware `__popc(a ^ b)` intrinsic for 32-bit XOR popcount, and assign **1 Warp (32 threads) per RANSAC hypothesis** where all 32 threads cooperatively evaluate inliers and reduce the count via `__shfl_down_sync`.
   - *Risk:* Register pressure in the 4-point DLT $8\times 8$ solver (`double` arrays spilling to local memory); mitigated by using single-precision `float` after Hartley normalization or having thread lane 0 solve the $8\times 8$ system in shared memory while all 32 lanes parallelize the inlier counting loop.
3. **Optimization 3 (`V4` – Multi-Stream Image A/B Concurrency + Pinned Memory + Texture Warping):**
   - *Idea:* Execute Stages 1–3 for Image A and Image B concurrently on two separate `cudaStream_t` streams, allocate host images in page-locked memory (`cudaMallocHost`) for asynchronous DMA transfers, and bind Image B to a 2D `cudaTextureObject_t` to leverage hardware bilinear interpolation and 2D spatial texture cache in Stage 6.
   - *Risk:* Page-locked memory allocation overhead (`cudaMallocHost`) if included inside timed sections, and diminished stream overlap if individual kernels already saturate all GPU SMs at 4K resolution.